# 2. Machine Learning for Regression

In [ ]:
import pandas as pd
import numpy as np

## 2.2 Data preparation

In [171]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv'

In [172]:
df = pd.read_csv(data)
df.head()

,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


In [173]:
df.columns

Index(['Make', 'Model', 'Year', 'Engine Fuel Type', 'Engine HP',
       'Engine Cylinders', 'Transmission Type', 'Driven_Wheels',
       'Number of Doors', 'Market Category', 'Vehicle Size', 'Vehicle Style',
       'highway MPG', 'city mpg', 'Popularity', 'MSRP'],
      dtype='str')

In [174]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.dtypes[df.dtypes == 'str']

In [ ]:
df.dtypes[df.dtypes == 'str'].index

In [ ]:
strings = list(df.dtypes[df.dtypes == 'str'].index)
strings

In [ ]:
for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')

In [ ]:
df.dtypes

## 2.3 Exploratory data analysis

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print()

In [ ]:
df

### Distribution of price

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
sns.histplot(df.msrp, bins=50)

In [ ]:
sns.histplot(df.msrp[df.msrp < 100000], bins=50)

### Explanation
The data has a very long tail to the right.
When the data is zoomed to max 100k, it shows that there are some prices with lots of 1k, while the other showed a normal one.
This is not a very good data, but we expect the data will be like this.
To overcome this type of distribution, use log.
Remember, log can't transform 0 value. That's why we use ***np.log1p*** functions and not ***np.log***

In [ ]:
np.log([1, 10, 1000, 100000])

In [ ]:
np.log([0, 1, 10, 1000, 100000])

In [ ]:
np.log([0+1, 1+1, 10+1, 1000+1, 100000+1])

In [ ]:
np.log1p([0, 1, 10, 100, 100000])

In [ ]:
price_logs = np.log1p(df.msrp)
price_logs

In [ ]:
sns.histplot(price_logs, bins=50)

The shape now resembles more like a bell-curve shape

### Missing values

In [ ]:
df.head()

In [ ]:
df.isna().sum()

## 2.4. Setting up the validation framework

In [ ]:
n = len(df) 
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

In [ ]:
n_val, n_test, n_train

In [ ]:
df_val = df.iloc[:n_val]
df_test = df.iloc[n_val:n_val+n_test]
df_train = df.iloc[n_val+n_test:]

In [ ]:
df_val

In [ ]:
df_train

In [ ]:
df_test

If the way we split our data is like that above, our data will not random. we need to shuffle the data, which the way to do that is as below.

In [ ]:
idx = np.arange(n)

In [ ]:
np.random.seed(2)
np.random.shuffle(idx)

In [ ]:
idx

In [ ]:
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train+n_val]]
df_test = df.iloc[idx[n_train+n_val:]]

In [ ]:
df_train.head()

In [ ]:
len(df_train), len(df_val), len(df_test)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
y_train = np.log1p(df_train.msrp.values)
y_val = np.log1p(df_val.msrp.values)
y_test = np.log1p(df_test.msrp.values)

In [ ]:
del df_train['msrp']
del df_val['msrp']
del df_test['msrp']

In [ ]:
df_train.head()

In [ ]:
len(y_train)

## 2.5 Linear regression

In [ ]:
df_train.head()

In [ ]:
df_train.iloc[10]

In [ ]:
xi = [453, 11, 86]

In [ ]:
w0 = 7.17
w = [0.01, 0.04, 0.002]

In [ ]:
def linear_regression(xi):
    n = len(xi)

    pred = w0

    for j in range(n):
        pred = pred + w[j] * xi[j]

    return pred

In [ ]:
linear_regression(xi)

In [ ]:
np.expm1(12.312)

## 2.6 Linear regression vector form

In [ ]:
def dot(xi, w):
    n = len(xi)

    res = 0.0

    for j in range(n):
        res = res + w[j] * xi[j]

    return res 

In [ ]:
def linear_regression(xi):
    return w0 + dot(xi, w)

In [ ]:
w_new = [w0] + w

In [ ]:
w_new

In [ ]:
def linear_regression(xi):
    xi = [1] + xi
    return dot(xi, w_new)

In [ ]:
linear_regression(xi)

In [ ]:
xi = [453, 11, 86]
w0 = 7.17
w = [0.01, 0.04, 0.002]
w_new = [w0] + w

In [ ]:
x1 = [1, 148, 24, 1385]
x2 = [1, 132, 25, 2031]
x10 = [1, 453, 11, 86]

X = [x1, x2,x10]
X = np.array(X)
X

In [ ]:
def linear_regression(X):
    return X.dot(w_new)

In [ ]:
linear_regression(X)

## 2.7 Training a linear regression model

In [ ]:
def train_linear_regression(X, y):
    pass

In [ ]:
X = [
    [148, 24, 1385],
    [132, 25, 2031],
    [453, 11, 86],
    [158, 24, 185],
    [172, 25, 201],
    [413, 11, 86],
    [38, 54, 185],
    [142, 25, 431],
    [453, 31, 86],
]
X = np.array(X)
X

In [ ]:
ones = np.ones(X.shape[0])
ones

In [ ]:
X = np.column_stack([ones, X])

In [ ]:
y = [10000, 20000, 15000, 20050, 10000, 20000, 15000, 25000, 12000]

In [ ]:
XTX = X.T.dot(X)
XTX

In [ ]:
XTX_inv = np.linalg.inv(XTX)

In [ ]:
w_full = XTX_inv.dot(X.T).dot(y)

In [ ]:
w0 = w_full[0]
w = w_full[1:]

In [ ]:
w0, w

In [ ]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [ ]:
train_linear_regression(X, y)

## 2.8 Car price baseline model

In [ ]:
df_train.dtypes

In [ ]:
df_train.columns

In [ ]:
base = ['engine_hp', 'engine_cylinders', 'highway_mpg',
        'city_mpg', 'popularity']

In [ ]:
df_train[base].head()

In [ ]:
df_train[base].values

In [ ]:
X_train = df_train[base].values

In [ ]:
y_train

In [ ]:
train_linear_regression(X_train, y_train)

*Nan because of missing value*

*So we need to fill the missing value*

In [ ]:
df_train[base].isna().sum()

In [ ]:
df_train[base].fillna(0).isna().sum()

In [ ]:
X_train = df_train[base].fillna(0).values

In [ ]:
train_linear_regression(X_train, y_train)

## Now success

In [ ]:
w0, w = train_linear_regression(X_train, y_train)

In [ ]:
y_pred = w0 + X_train.dot(w)

In [ ]:
sns.histplot(y_pred, color='red', alpha=0.5, bins = 50)
sns.histplot(y_train, color='blue', alpha=0.5, bins = 50)

The results shown that the model is not ideal, since the prediction (red) less than the target (blue)

# 2. Machine Learning for Regression

In [ ]:
import pandas as pd
import numpy as np

## 2.2 Data preparation

In [ ]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv'

In [ ]:
df = pd.read_csv(data)
df.head()

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.dtypes[df.dtypes == 'str']

In [ ]:
df.dtypes[df.dtypes == 'str'].index

In [ ]:
strings = list(df.dtypes[df.dtypes == 'str'].index)
strings

In [ ]:
for col in strings:
    df[col] = df[col].str.lower().str.replace(' ', '_')

In [ ]:
df.dtypes

## 2.3 Exploratory data analysis

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
for col in df.columns:
    print(col)
    print(df[col].unique()[:5])
    print(df[col].nunique())
    print()

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
sns.histplot(df.msrp, bins=50)

In [ ]:
sns.histplot(df.msrp[df.msrp < 100000], bins=50)

In [ ]:
np.log([1, 10, 1000, 100000])

In [ ]:
np.log([0, 1, 10, 1000, 100000])

In [ ]:
np.log([0+1, 1+1, 10+1, 1000+1, 100000+1])

In [ ]:
np.log1p([0, 1, 10, 100, 100000])

In [ ]:
price_logs = np.log1p(df.msrp)
price_logs

In [ ]:
sns.histplot(price_logs, bins=50)

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
n = len(df) 
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

In [ ]:
n_val, n_test, n_train

In [ ]:
df_val = df.iloc[:n_val]
df_test = df.iloc[n_val:n_val+n_test]
df_train = df.iloc[n_val+n_test:]

In [ ]:
df_val

In [ ]:
df_train

In [ ]:
df_test

In [ ]:
idx = np.arange(n)

In [ ]:
np.random.seed(2)
np.random.shuffle(idx)

In [ ]:
idx

In [ ]:
df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train+n_val]]
df_test = df.iloc[idx[n_train+n_val:]]

In [ ]:
df_train.head()

In [ ]:
len(df_train), len(df_val), len(df_test)

In [ ]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [ ]:
y_train = np.log1p(df_train.msrp.values)
y_val = np.log1p(df_val.msrp.values)
y_test = np.log1p(df_test.msrp.values)

In [ ]:
del df_train['msrp']
del df_val['msrp']
del df_test['msrp']

In [ ]:
df_train.head()

In [ ]:
len(y_train)

In [ ]:
df_train.head()

In [ ]:
df_train.iloc[10]

In [ ]:
xi = [453, 11, 86]

In [ ]:
w0 = 7.17
w = [0.01, 0.04, 0.002]

In [ ]:
def linear_regression(xi):
    n = len(xi)

    pred = w0

    for j in range(n):
        pred = pred + w[j] * xi[j]

    return pred

In [ ]:
linear_regression(xi)

In [ ]:
np.expm1(12.312)

In [ ]:
def dot(xi, w):
    n = len(xi)

    res = 0.0

    for j in range(n):
        res = res + w[j] * xi[j]

    return res 

In [ ]:
def linear_regression(xi):
    return w0 + dot(xi, w)

In [ ]:
w_new = [w0] + w

In [ ]:
w_new

In [ ]:
def linear_regression(xi):
    xi = [1] + xi
    return dot(xi, w_new)

In [ ]:
linear_regression(xi)

In [ ]:
xi = [453, 11, 86]
w0 = 7.17
w = [0.01, 0.04, 0.002]
w_new = [w0] + w

In [ ]:
x1 = [1, 148, 24, 1385]
x2 = [1, 132, 25, 2031]
x10 = [1, 453, 11, 86]

X = [x1, x2,x10]
X = np.array(X)
X

In [ ]:
def linear_regression(X):
    return X.dot(w_new)

In [ ]:
linear_regression(X)

In [ ]:
def train_linear_regression(X, y):
    pass

In [ ]:
X = [
    [148, 24, 1385],
    [132, 25, 2031],
    [453, 11, 86],
    [158, 24, 185],
    [172, 25, 201],
    [413, 11, 86],
    [38, 54, 185],
    [142, 25, 431],
    [453, 31, 86],
]
X = np.array(X)
X

In [ ]:
ones = np.ones(X.shape[0])
ones

In [ ]:
X = np.column_stack([ones, X])

In [ ]:
y = [10000, 20000, 15000, 20050, 10000, 20000, 15000, 25000, 12000]

In [ ]:
XTX = X.T.dot(X)
XTX

In [ ]:
XTX_inv = np.linalg.inv(XTX)

In [ ]:
w_full = XTX_inv.dot(X.T).dot(y)

In [ ]:
w0 = w_full[0]
w = w_full[1:]

In [ ]:
w0, w

In [ ]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    
    return w_full[0], w_full[1:]

In [ ]:
train_linear_regression(X, y)

In [ ]:
df_train.dtypes

In [ ]:
df_train.columns

In [ ]:
base = ['engine_hp', 'engine_cylinders', 'highway_mpg',
        'city_mpg', 'popularity']

In [ ]:
df_train[base].head()

In [ ]:
df_train[base].values

In [ ]:
X_train = df_train[base].values

In [ ]:
y_train

In [ ]:
train_linear_regression(X_train, y_train)

In [ ]:
df_train[base].isna().sum()

In [ ]:
df_train[base].fillna(0).isna().sum()

In [ ]:
X_train = df_train[base].fillna(0).values

In [ ]:
train_linear_regression(X_train, y_train)

## Now success

In [ ]:
w0, w = train_linear_regression(X_train, y_train)

In [ ]:
y_pred = w0 + X_train.dot(w)

In [ ]:
sns.histplot(y_pred, color='red', alpha=0.5, bins = 50)
sns.histplot(y_train, color='blue', alpha=0.5, bins = 50)

## 2.9 RMSE

In [ ]:
def rmse(y, y_pred):
    error = y - y_pred
    se = error ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [ ]:
rmse(y_train, y_pred)

## 2.10 Validating the model

In [ ]:
base = ['engine_hp', 'engine_cylinders', 'highway_mpg',
        'city_mpg', 'popularity']

X_train = df_train[base].fillna(0).values

w0, w = train_linear_regression(X_train, y_train)

y_pred = w0 + X_train.dot(w)

In [ ]:
def prepare_X(df):
    df_num = df[base]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

## 2.11 Simple feature engineering

In [ ]:
df_train.head()

In [ ]:
df_train.year.max()

In [ ]:
2017 - df_train.year

In [ ]:
def prepare_X(df):
    df = df.copy()
    
    df['age'] = 2017 - df.year
    features = base + ['age']
    
    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    return X

In [ ]:
X_train = prepare_X(df_train)

In [ ]:
df_train.columns

In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

In [ ]:
sns.histplot(y_pred, color='red', alpha=0.5, bins = 50)
sns.histplot(y_val, color='blue', alpha=0.5, bins = 50)

Look at the RMSE, lower error means better model. RMSE become 0.51, much better model now. Histogram also shows improvement

## 2.12 Categorical variables

In [ ]:
df_train.head()

In [ ]:
df_train.dtypes

In [ ]:
df_train.number_of_doors

In [ ]:
#df_train['num_doors_2'] = (df_train.number_of_doors == 2).astype('int')
#df_train['num_doors_3'] = (df_train.number_of_doors == 3).astype('int')
#df_train['num_doors_4'] = (df_train.number_of_doors == 4).astype('int')

or, we can use looping

In [ ]:
for v in [2, 3, 4]:
    df_train['num_doors_%s' % v] = (df_train.number_of_doors == v).astype('int')

In [ ]:
df_train.head()

In [ ]:
def prepare_X(df):
    df = df.copy()
    features = base.copy()
    
    df['age'] = 2017 - df.year
    features.append('age')

    for v in[2, 3, 4]:
        df['num_doors_%s' % v] = (df.number_of_doors == v).astype('int')
        features.append('num_doors_%s' % v)
    
    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    
    return X

In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

In [ ]:
df.make.value_counts().head()

In [ ]:
makes = list(df.make.value_counts().head().index)
makes

In [ ]:
def prepare_X(df):
    df = df.copy()
    features = base.copy()
    
    df['age'] = 2017 - df.year
    features.append('age')

    for v in[2, 3, 4]:
        df['num_doors_%s' % v] = (df.number_of_doors == v).astype('int')
        features.append('num_doors_%s' % v)

    for v in makes:
        df['make_%s' % v] = (df.make == v).astype('int')
        features.append('make_%s' % v)
    
    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    
    return X

In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

In [ ]:
df_train.dtypes

In [ ]:
categorical_variables = [
    'make', 'engine_fuel_type', 'transmission_type', 'driven_wheels',
    'market_category', 'vehicle_size', 'vehicle_style'
]
categorical_variables

In [ ]:
categories = {}

for c in categorical_variables:
    categories[c] = list(df[c].value_counts().head().index)
categories

In [ ]:
def prepare_X(df):
    df = df.copy()
    features = base.copy()
    
    df['age'] = 2017 - df.year
    features.append('age')

    for v in[2, 3, 4]:
        df['num_doors_%s' % v] = (df.number_of_doors == v).astype('int')
        features.append('num_doors_%s' % v)

    for c, values in categories.items():
        for v in makes:
            df['%s_%s' % (c, v)] = (df[c] == v).astype('int')
            features.append('%s_%s' % (c, v))
    
    df_num = df[features]
    df_num = df_num.fillna(0)
    X = df_num.values
    
    return X

In [ ]:
X_train = prepare_X(df_train)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)

rmse(y_val, y_pred)

## 2.13 Regularization